# Season-Aggregate Team Forecast -- Kaggle GPU Run

**Track**: season-aggregate (deep history). Targets: `win_pct`,
`runs_scored_per_game`, `runs_allowed_per_game`, `team_ops`, `team_era`.
All 30 teams, pooled into one shared `TeamPanelNet` (team identity as a
learned embedding) -- see `../../season_aggregate_smoke.ipynb` for the
CPU-scale pipeline-correctness check this scales up from, and the plan
document ("PyTorch Team-Forecast Panel Models") for full rationale.

**Split**: train on years strictly before 2015 (~114 years x up to 30
teams). Holdout: ALL of 2015-2025, never trained on -- a genuine backtest
of whether pre-Statcast-era pattern-learning predicts the modern game.

**Kaggle settings required** (see `kernel-metadata.json`): GPU accelerator
(T4x2) and internet access both enabled -- this notebook fetches its own
training data live from the MLB Stats API.


In [ ]:

import json
import os
import time
import numpy as np
import pandas as pd
import requests
import torch
from sklearn.metrics import mean_squared_error, r2_score

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

BASE_URL = "https://statsapi.mlb.com/api/v1"
HEADERS = {"User-Agent": "Mozilla/5.0 (MLB-Analytics-Dashboard-Telemetry/1.0; AustinKuo)"}


def safe_float(val, default=float("nan")):
    try:
        s = str(val).strip()
        return default if s in ("-.--", "---", "", "INF", "inf") else float(s)
    except (ValueError, TypeError):
        return default


def safe_int(val, default=0):
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return default

print("Environment ready.")


In [ ]:

TRACK = "season_aggregate"
ENVIRONMENT = "kaggle_gpu"

ALL_TEAMS = [{'team_id': 108, 'abbreviation': 'LAA', 'founding_year': 1961}, {'team_id': 109, 'abbreviation': 'ARI', 'founding_year': 1998}, {'team_id': 110, 'abbreviation': 'BAL', 'founding_year': 1901}, {'team_id': 111, 'abbreviation': 'BOS', 'founding_year': 1901}, {'team_id': 112, 'abbreviation': 'CHC', 'founding_year': 1901}, {'team_id': 113, 'abbreviation': 'CIN', 'founding_year': 1901}, {'team_id': 114, 'abbreviation': 'CLE', 'founding_year': 1901}, {'team_id': 115, 'abbreviation': 'COL', 'founding_year': 1993}, {'team_id': 116, 'abbreviation': 'DET', 'founding_year': 1901}, {'team_id': 117, 'abbreviation': 'HOU', 'founding_year': 1962}, {'team_id': 118, 'abbreviation': 'KC', 'founding_year': 1969}, {'team_id': 119, 'abbreviation': 'LAD', 'founding_year': 1901}, {'team_id': 120, 'abbreviation': 'WSH', 'founding_year': 1969}, {'team_id': 121, 'abbreviation': 'NYM', 'founding_year': 1962}, {'team_id': 133, 'abbreviation': 'ATH', 'founding_year': 1901}, {'team_id': 134, 'abbreviation': 'PIT', 'founding_year': 1901}, {'team_id': 135, 'abbreviation': 'SD', 'founding_year': 1969}, {'team_id': 136, 'abbreviation': 'SEA', 'founding_year': 1977}, {'team_id': 137, 'abbreviation': 'SF', 'founding_year': 1901}, {'team_id': 138, 'abbreviation': 'STL', 'founding_year': 1901}, {'team_id': 139, 'abbreviation': 'TB', 'founding_year': 1998}, {'team_id': 140, 'abbreviation': 'TEX', 'founding_year': 1961}, {'team_id': 141, 'abbreviation': 'TOR', 'founding_year': 1977}, {'team_id': 142, 'abbreviation': 'MIN', 'founding_year': 1901}, {'team_id': 143, 'abbreviation': 'PHI', 'founding_year': 1901}, {'team_id': 144, 'abbreviation': 'ATL', 'founding_year': 1901}, {'team_id': 145, 'abbreviation': 'CWS', 'founding_year': 1901}, {'team_id': 146, 'abbreviation': 'MIA', 'founding_year': 1993}, {'team_id': 147, 'abbreviation': 'NYY', 'founding_year': 1901}, {'team_id': 158, 'abbreviation': 'MIL', 'founding_year': 1969}]
TEAM_FOUNDING_YEAR = {t["team_id"]: t["founding_year"] for t in ALL_TEAMS}

TRAIN_START_YEAR = 1901
TRAIN_END_YEAR = 2014
HOLDOUT_START_YEAR = 2015
HOLDOUT_END_YEAR = 2025
FORECAST_END_YEAR = 2027

TARGETS = ["win_pct", "runs_scored_per_game", "runs_allowed_per_game", "team_ops", "team_era"]

HIDDEN = (64, 32)
EMBED_DIM = 8
DROPOUT = 0.2
EPOCHS = 500
EARLY_STOPPING_PATIENCE = 25
LR = 1e-3
WEIGHT_DECAY = 1e-4

print(f"{len(ALL_TEAMS)} teams, train<= {TRAIN_END_YEAR}, holdout {HOLDOUT_START_YEAR}-{HOLDOUT_END_YEAR}")


## Data fetch (all 30 teams, full history)

Same vendored fetch as the smoke notebook -- reimplements
`macroservice/teams.py::get_team_season_stats` inline so this notebook
stays self-contained on Kaggle. At 30 teams x up to ~124 years x 2 stat
groups this is thousands of requests; expect this cell to take several
minutes.


In [ ]:

def fetch_team_season_stat_row(team_id, year):
    try:
        hit = requests.get(f"{BASE_URL}/teams/{team_id}/stats",
                            params={"stats": "season", "group": "hitting", "season": year},
                            headers=HEADERS, timeout=15).json()
        pit = requests.get(f"{BASE_URL}/teams/{team_id}/stats",
                            params={"stats": "season", "group": "pitching", "season": year},
                            headers=HEADERS, timeout=15).json()
    except Exception:
        return None
    hit_splits = hit.get("stats", [{}])[0].get("splits", [])
    pit_splits = pit.get("stats", [{}])[0].get("splits", [])
    if not hit_splits or not pit_splits:
        return None
    h, p = hit_splits[0]["stat"], pit_splits[0]["stat"]
    games = safe_int(p.get("gamesPlayed"))
    if games == 0:
        return None
    return {"games": games, "wins": safe_int(p.get("wins")), "runs_scored": safe_int(h.get("runs")),
            "runs_allowed": safe_int(p.get("runs")), "team_ops": safe_float(h.get("ops")),
            "team_era": safe_float(p.get("era"))}


def fetch_season_aggregate_frame(teams, start_year, end_year):
    rows = []
    for team in teams:
        floor = max(start_year, TEAM_FOUNDING_YEAR.get(team["team_id"], start_year))
        for year in range(floor, end_year + 1):
            row = fetch_team_season_stat_row(team["team_id"], year)
            if row is None:
                continue
            row.update(team_id=team["team_id"], year=year)
            rows.append(row)
    df = pd.DataFrame(rows)
    df["win_pct"] = df["wins"] / df["games"]
    df["runs_scored_per_game"] = df["runs_scored"] / df["games"]
    df["runs_allowed_per_game"] = df["runs_allowed"] / df["games"]
    return df.sort_values(["team_id", "year"]).reset_index(drop=True)


t0 = time.time()
raw = fetch_season_aggregate_frame(ALL_TEAMS, TRAIN_START_YEAR, HOLDOUT_END_YEAR)
n_teams_fetched = raw["team_id"].nunique()
print(f"Fetched {len(raw)} team-year rows across {n_teams_fetched} teams in {time.time()-t0:.0f}s.")


In [ ]:

def add_lag_rolling_features(df, targets):
    df = df.sort_values(["team_id", "year"]).copy()
    for target in targets:
        grp = df.groupby("team_id")[target]
        df[f"lag_1_{target}"] = grp.shift(1)
        df[f"lag_3_avg_{target}"] = grp.shift(1).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
        df[f"rolling_5yr_{target}"] = grp.shift(1).rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
    return df


team_ids_sorted = sorted(t["team_id"] for t in ALL_TEAMS)
TEAM_EMBED_INDEX = {tid: i for i, tid in enumerate(team_ids_sorted)}

featured = add_lag_rolling_features(raw, TARGETS)
featured["team_embedding_index"] = featured["team_id"].map(TEAM_EMBED_INDEX)
featured["year_norm"] = (featured["year"] - 1901) / 125.0
featured["team_age"] = featured["year"] - featured["team_id"].map(TEAM_FOUNDING_YEAR)

FEATURE_COLS = ["year_norm", "team_age"] + [
    f"{prefix}_{t}" for t in TARGETS for prefix in ("lag_1", "lag_3_avg", "rolling_5yr")
]
featured = featured.dropna(subset=FEATURE_COLS).reset_index(drop=True)
print(f"{len(featured)} feature-complete rows, {len(FEATURE_COLS)} numeric features.")


## Split, model, training

Internal validation for early stopping: last 2 pre-2015 years per team --
never the real 2015-2025 holdout.


In [ ]:

import torch
import torch.nn as nn


class TeamPanelNet(nn.Module):
    """Feedforward net over engineered lag/rolling features, with team
    identity as a learned embedding and one shared trunk predicting all
    targets at once (a regularizer in itself at this row count, since the
    targets are correlated -- wins/runs-scored/runs-allowed/OPS/ERA all
    move together).
    """

    def __init__(self, n_teams, n_numeric_features, n_targets, embed_dim=8, hidden=(64, 32), dropout=0.2):
        super().__init__()
        self.team_embed = nn.Embedding(n_teams, embed_dim)
        layers, in_dim = [], embed_dim + n_numeric_features
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        self.trunk = nn.Sequential(*layers)
        self.head = nn.Linear(in_dim, n_targets)

    def forward(self, team_idx, x_numeric):
        z = torch.cat([self.team_embed(team_idx), x_numeric], dim=-1)
        return self.head(self.trunk(z))


def train_panel_net(model, X_team, X_num, y, epochs, lr=1e-3, weight_decay=1e-4,
                     val_team=None, val_num=None, val_y=None, patience=25):
    """Adam + early stopping on an internal validation set the caller
    carves out (never the real holdout -- see each notebook's split cell).
    Returns (train_loss_curve, val_loss_curve); leaves `model` trained
    in-place at its best-validation-loss state.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = torch.nn.MSELoss()
    train_curve, val_curve = [], []
    best_val, best_state, bad_epochs = float("inf"), None, 0
    has_val = val_team is not None and len(val_team) > 0

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X_team, X_num)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        train_curve.append(float(loss.item()))

        if has_val:
            model.eval()
            with torch.no_grad():
                val_loss = float(loss_fn(model(val_team, val_num), val_y).item())
            val_curve.append(val_loss)
            if val_loss < best_val:
                best_val, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    break

    if has_val and best_state is not None:
        model.load_state_dict(best_state)
    return train_curve, val_curve


In [ ]:

train_mask = featured["year"] <= TRAIN_END_YEAR
holdout_mask = (featured["year"] >= HOLDOUT_START_YEAR) & (featured["year"] <= HOLDOUT_END_YEAR)
train_df = featured[train_mask].copy()
holdout_df = featured[holdout_mask].copy()

val_years_per_team = train_df.groupby("team_id")["year"].apply(lambda s: set(s.nlargest(2)))
is_val = train_df.apply(lambda r: r["year"] in val_years_per_team.get(r["team_id"], set()), axis=1)
fit_df, val_df = train_df[~is_val], train_df[is_val]

feat_mean, feat_std = fit_df[FEATURE_COLS].mean(), fit_df[FEATURE_COLS].std().replace(0, 1.0)
y_mean, y_std = fit_df[TARGETS].mean(), fit_df[TARGETS].std().replace(0, 1.0)


def to_tensors(df):
    team_idx = torch.tensor(df["team_embedding_index"].to_numpy(), dtype=torch.long, device=DEVICE)
    x_num = torch.tensor(((df[FEATURE_COLS] - feat_mean) / feat_std).to_numpy(), dtype=torch.float32, device=DEVICE)
    y = torch.tensor(((df[TARGETS] - y_mean) / y_std).to_numpy(), dtype=torch.float32, device=DEVICE)
    return team_idx, x_num, y


fit_team, fit_num, fit_y = to_tensors(fit_df)
val_team, val_num, val_y = to_tensors(val_df) if len(val_df) else (None, None, None)

model = TeamPanelNet(len(team_ids_sorted), len(FEATURE_COLS), len(TARGETS),
                      embed_dim=EMBED_DIM, hidden=HIDDEN, dropout=DROPOUT).to(DEVICE)
t0 = time.time()
train_curve, val_curve = train_panel_net(model, fit_team, fit_num, fit_y, epochs=EPOCHS, lr=LR,
                                          weight_decay=WEIGHT_DECAY, val_team=val_team, val_num=val_num,
                                          val_y=val_y, patience=EARLY_STOPPING_PATIENCE)
print(f"Trained {len(train_curve)} epochs in {time.time() - t0:.1f}s. Final train loss={train_curve[-1]:.4f}")


## Holdout evaluation + sanity baseline

In [ ]:

model.eval()
with torch.no_grad():
    holdout_team, holdout_num, _ = to_tensors(holdout_df.assign(**{t: 0.0 for t in TARGETS}))
    pred_scaled = model(holdout_team, holdout_num).cpu().numpy()
predictions = pred_scaled * y_std.to_numpy() + y_mean.to_numpy()

holdout_predictions, aggregate_holdout_metrics = [], {}
for i, target in enumerate(TARGETS):
    actual = holdout_df[target].to_numpy()
    pred = predictions[:, i]
    valid = ~np.isnan(actual)
    r2 = float(r2_score(actual[valid], pred[valid])) if valid.sum() >= 2 else None
    rmse = float(np.sqrt(mean_squared_error(actual[valid], pred[valid]))) if valid.sum() >= 2 else None
    aggregate_holdout_metrics[target] = {"r2": r2, "rmse": rmse, "n": int(valid.sum())}
    for (_, row), a, p in zip(holdout_df.iterrows(), actual, pred):
        holdout_predictions.append({"team_id": int(row["team_id"]), "year": int(row["year"]),
                                     "metric": target, "actual": None if np.isnan(a) else float(a),
                                     "predicted": float(p)})
    print(f"{target:>24s}  holdout R2={r2}  RMSE={rmse}  n={valid.sum()}")

from sklearn.linear_model import LinearRegression
baseline_comparison = {}
for target in TARGETS:
    lr_model = LinearRegression().fit(fit_df[FEATURE_COLS], fit_df[target])
    pred = lr_model.predict(holdout_df[FEATURE_COLS])
    actual = holdout_df[target].to_numpy()
    valid = ~np.isnan(actual)
    baseline_comparison[target] = {
        "r2": float(r2_score(actual[valid], pred[valid])) if valid.sum() >= 2 else None,
        "rmse": float(np.sqrt(mean_squared_error(actual[valid], pred[valid]))) if valid.sum() >= 2 else None,
    }
print("Baseline (plain linear regression) holdout R2:", {k: v["r2"] for k, v in baseline_comparison.items()})


## Forward forecast

In [ ]:

def forecast_forward(model, history_df, team_id, start_year, end_year):
    history = history_df[history_df["team_id"] == team_id].sort_values("year").copy()
    out = []
    for year in range(start_year, end_year + 1):
        row = {"team_id": team_id, "year": year, "year_norm": (year - 1901) / 125.0,
               "team_age": year - TEAM_FOUNDING_YEAR.get(team_id, year)}
        recent = history.tail(5)
        for target in TARGETS:
            row[f"lag_1_{target}"] = history[target].iloc[-1]
            row[f"lag_3_avg_{target}"] = history[target].tail(3).mean()
            row[f"rolling_5yr_{target}"] = recent[target].mean()
        x_num = torch.tensor([[(row[c] - feat_mean[c]) / feat_std[c] for c in FEATURE_COLS]],
                              dtype=torch.float32, device=DEVICE)
        team_idx = torch.tensor([TEAM_EMBED_INDEX[team_id]], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            pred = (model(team_idx, x_num).cpu().numpy()[0] * y_std.to_numpy() + y_mean.to_numpy())
        for i, target in enumerate(TARGETS):
            row[target] = float(pred[i])
        out.append(row)
        history = pd.concat([history, pd.DataFrame([row])], ignore_index=True)
    return out


forward_forecasts = []
for team in ALL_TEAMS:
    for row in forecast_forward(model, featured, team["team_id"], HOLDOUT_END_YEAR + 1, FORECAST_END_YEAR):
        for target in TARGETS:
            forward_forecasts.append({"team_id": team["team_id"], "year": row["year"], "metric": target,
                                       "predicted": row[target], "ci_lower": None, "ci_upper": None})
print(f"{len(forward_forecasts)} forward forecast rows ({HOLDOUT_END_YEAR + 1}-{FORECAST_END_YEAR}).")


## Results JSON export

Download this from `/kaggle/working/` after the run completes, commit it
into `notebooks/results/` in the repo, then run
`python scripts/load_team_forecasts.py notebooks/results/<file>.json`
to populate Postgres.


In [ ]:

import datetime
import subprocess

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
except Exception:
    git_commit = None

run_stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y.%m.%d-%H%M")
results = {
    "schema_version": "1.0",
    "track": TRACK,
    "model_version": f"{TRACK}-{run_stamp}-gpu",
    "run_timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "environment": ENVIRONMENT,
    "git_commit": git_commit,
    "random_seed": 42,
    "hyperparameters": {"hidden_dims": list(HIDDEN), "embedding_dim": EMBED_DIM, "dropout": DROPOUT,
                         "lr": LR, "weight_decay": WEIGHT_DECAY, "epochs_trained": len(train_curve),
                         "early_stopping_patience": EARLY_STOPPING_PATIENCE},
    "targets": TARGETS,
    "feature_list": FEATURE_COLS,
    "teams": [{"team_id": t["team_id"], "abbreviation": t["abbreviation"],
               "embedding_index": TEAM_EMBED_INDEX[t["team_id"]]} for t in ALL_TEAMS],
    "training_window": {"start_year": TRAIN_START_YEAR, "end_year": TRAIN_END_YEAR},
    "holdout_window": {"start_year": HOLDOUT_START_YEAR, "end_year": HOLDOUT_END_YEAR},
    "regime_flags_used": [],
    "excluded_rows": [],
    "baseline_comparison": baseline_comparison,
    "aggregate_holdout_metrics": aggregate_holdout_metrics,
    "holdout_predictions": holdout_predictions,
    "forward_forecasts": forward_forecasts,
    "loss_curve": {"train": train_curve, "val": val_curve},
    "notes": "",
}

out_dir = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/{results['model_version']}.json"
with open(out_path, "w") as fh:
    json.dump(results, fh, indent=2)
print("Wrote", out_path)
